In [51]:
import pandas as pd

In [159]:
import numpy as np
df = pd.read_csv(r"C:\Users\user\Downloads\online_retail.csv")

In [160]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [161]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12-01-2010,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12-01-2010,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12-01-2010,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12-01-2010,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12-01-2010,3.39,17850.0,United Kingdom


In [162]:
df.isnull()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
541904,False,False,False,False,False,False,False,False
541905,False,False,False,False,False,False,False,False
541906,False,False,False,False,False,False,False,False
541907,False,False,False,False,False,False,False,False


In [163]:
df.dtypes

InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object

In [164]:
df = pd.read_csv(
    "online_retail.csv",
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "Quantity": "Int16",
        "UnitPrice": "float32"
    }
)

In [165]:
df.dtypes

InvoiceNo       string
StockCode          str
Description        str
Quantity         Int16
InvoiceDate        str
UnitPrice      float32
CustomerID     float64
Country            str
dtype: object

In [166]:
df["CustomerID"] = df["CustomerID"].astype("Int32")

In [167]:
df.dtypes

InvoiceNo       string
StockCode          str
Description        str
Quantity         Int16
InvoiceDate        str
UnitPrice      float32
CustomerID       Int32
Country            str
dtype: object

In [168]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [169]:
df= df.dropna(subset=["CustomerID"])

In [170]:
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [171]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12-01-2010,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12-01-2010,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12-01-2010,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12-01-2010,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12-01-2010,3.39,17850,United Kingdom


In [172]:
df=df[df["Quantity"] > 0] #actual purchase
df=df[df["UnitPrice"] > 0] #real revenue

In [173]:
df["Amount"] = df["Quantity"] * df["UnitPrice"]

In [174]:
df.dtypes

InvoiceNo       string
StockCode          str
Description        str
Quantity         Int16
InvoiceDate        str
UnitPrice      float32
CustomerID       Int32
Country            str
Amount         Float32
dtype: object

In [175]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Amount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12-01-2010,2.55,17850,United Kingdom,15.299999
1,536365,71053,WHITE METAL LANTERN,6,12-01-2010,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12-01-2010,2.75,17850,United Kingdom,22.0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12-01-2010,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12-01-2010,3.39,17850,United Kingdom,20.34


In [176]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="%m/%d/%Y %H:%M",
    errors="coerce"
)

In [177]:
import datetime as dt
snapshot_date = df["InvoiceDate"].max() + dt.timedelta(days=1)
print(snapshot_date)

2011-12-01 17:37:00


In [178]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df = df.dropna(subset=["CustomerID", "InvoiceDate"])

snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm_base = (
    df.groupby("CustomerID")
    .agg(
        Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("Amount", "sum")
    )
    .reset_index()
)

In [179]:
rfm_base.head()

,CustomerID,Recency,Frequency,Monetary
0,12346,317,1,9026.160156
1,12347,31,2,1769.709961
2,12348,67,3,1430.23999
3,12349,10,1,1757.550049
4,12352,64,5,1209.660034


In [180]:
rfm_base.describe()

,CustomerID,Recency,Frequency,Monetary
count,3480.0,3480.000000,3480.000000,3480.0
mean,15296.640805,92.294828,3.037644,1427.517944
std,1716.677413,96.081325,4.831631,5920.51709
min,12346.0,1.000000,1.000000,1.0
25%,13827.75,13.000000,1.000000,256.407501
50%,15300.0,49.000000,2.000000,541.495026
75%,16767.25,155.000000,3.000000,1226.35495
max,18287.0,353.000000,126.000000,192588.625


In [181]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Amount
26732,538521,21754,HOME BUILDING BLOCK WORD,3,2010-12-13 09:02:00,5.950000,14180,United Kingdom,17.849998
26733,538521,21755,LOVE BUILDING BLOCK WORD,3,2010-12-13 09:02:00,5.950000,14180,United Kingdom,17.849998
26734,538521,22072,RED RETROSPOT TEA CUP AND SAUCER,8,2010-12-13 09:02:00,3.750000,14180,United Kingdom,30.0
26735,538521,22846,BREAD BIN DINER STYLE RED,1,2010-12-13 09:02:00,16.950001,14180,United Kingdom,16.950001
26736,538521,22849,BREAD BIN DINER STYLE MINT,1,2010-12-13 09:02:00,16.950001,14180,United Kingdom,16.950001


In [182]:
#Recency Scoring 
rfm_base["R_score"]=pd.qcut(rfm_base["Recency"],
                      5,
                      labels=[5,4,3,2,1])

In [183]:
rfm_base["F_score"] = pd.qcut(rfm_base["Frequency"].rank(method="first"),
                             5,
                             labels=[1,2,3,4,5])

In [184]:
rfm_base["M_score"]= pd.qcut(rfm_base["Monetary"].rank(method="first"),
                            5,
                            labels=[1,2,3,4,5])

In [185]:
rfm_base.columns

Index(['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_score', 'F_score',
       'M_score'],
      dtype='str')

In [186]:
rfm_base["R_score"].isnull().sum()

np.int64(0)

In [187]:
rfm_base["F_score"].isnull().sum()
rfm_base["M_score"].isnull().sum()

np.int64(0)

In [188]:
rfm_base["R_score"] = rfm_base["R_score"].cat.add_categories([0]).fillna(0)
rfm_base["F_score"] = rfm_base["F_score"].cat.add_categories([0]).fillna(0)
rfm_base["M_score"] = rfm_base["M_score"].cat.add_categories([0]).fillna(0)

In [189]:
rfm_base["R_score"] = rfm_base["R_score"].astype(int)
rfm_base["F_score"] = rfm_base["F_score"].astype(int)
rfm_base["M_score"] = rfm_base["M_score"].astype(int)

In [190]:
def segment_customer(row):
    
    if row["R_score"] >= 4 and row["F_score"] >= 4 and row["M_score"] >= 4:
        return "Champion"
    
    elif row["R_score"] >= 3 and row["F_score"] >= 3:
        return "Loyal Customer"
    
    elif row["R_score"] >= 4:
        return "Recent Customer"
    
    elif row["R_score"] <= 2 and row["F_score"] >= 3:
        return "At Risk"
    
    else:
        return "Churn Risk"


In [191]:
rfm_base["Segment"] = rfm_base.apply(segment_customer, axis=1)

In [192]:
rfm_base[["CustomerID","R_score","F_score","M_score","Segment"]].head()


,CustomerID,R_score,F_score,M_score,Segment
0,12346,1,1,5,Churn Risk
1,12347,4,3,5,Loyal Customer
2,12348,3,4,4,Loyal Customer
3,12349,5,1,5,Recent Customer
4,12352,3,5,4,Loyal Customer


In [193]:
rfm_base["Segment"].value_counts()


Segment
Churn Risk         1088
Loyal Customer      821
Champion            682
At Risk             585
Recent Customer     304
Name: count, dtype: int64

In [194]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M")

cohort = (
    df.groupby("CustomerID")["InvoiceMonth"]
    .min()
    .reset_index()
)
cohort.columns = ["CustomerID", "CohortMonth"]

df = df.merge(cohort, on="CustomerID")

df["CohortIndex"] = (
    (df["InvoiceMonth"].dt.year - df["CohortMonth"].dt.year) * 12
    + (df["InvoiceMonth"].dt.month - df["CohortMonth"].dt.month)
    + 1
)

In [195]:
cohort_data = (
    df.groupby(["CohortMonth", "CohortIndex"])["CustomerID"]
      .nunique()
      .reset_index()
)


In [196]:
cohort_pivot = cohort_data.pivot(
    index="CohortMonth",
    columns="CohortIndex",
    values="CustomerID"
)

In [197]:
cohort_size = cohort_pivot.iloc[:, 0]

retention = cohort_pivot.divide(cohort_size, axis=0)


In [198]:
retention.head()


CohortIndex,1,2,3,4,5,6,7,8,9,10,11,12
CohortMonth,,,,,,,,,,,,
2010-12,1.0,0.354271,0.286432,0.361809,0.286432,0.371859,0.341709,0.311558,0.304020,0.329146,0.283920,0.449749
2011-01,1.0,0.198433,0.266319,0.193211,0.214099,0.229765,0.237598,0.206266,0.242820,0.266319,0.328982,NaN
2011-02,1.0,0.175159,0.165605,0.210191,0.210191,0.226115,0.194268,0.226115,0.242038,0.261146,NaN,NaN
2011-03,1.0,0.112272,0.201044,0.172324,0.201044,0.143603,0.219321,0.201044,0.240209,NaN,NaN,NaN
2011-04,1.0,0.179039,0.144105,0.152838,0.157205,0.196507,0.183406,0.187773,NaN,NaN,NaN,NaN


In [199]:
pip install ipykernel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [200]:
# plt.figure(figsize=(12, 8))

# sns.heatmap(
#     retention,
#     annot=True,
#     fmt=".2f",
#     cmap="Blues"
# )

# plt.title("Monthly Customer Retention Cohort Analysis")
# plt.xlabel("Cohort Index (Months Since First Purchase)")
# plt.ylabel("Cohort Month")

# plt.show()

In [201]:
!pip install mlxtend


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [202]:
from mlxtend.frequent_patterns import apriori, association_rules

In [203]:
basket = (
    df.groupby(['InvoiceNo', 'Description'])['Quantity']
      .sum()
      .unstack()
      .fillna(0)
)

In [204]:
basket = (basket > 0).astype(int)

In [205]:
basket = basket.astype(bool)

In [206]:
frequent_items = apriori(
    basket,
    min_support=0.02,
    use_colnames=True
)

In [207]:
# Rules
rules = association_rules(
    frequent_items,
    metric="lift",
    min_threshold=1
)

rules.sort_values("lift", ascending=False).head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
83,frozenset({GREEN REGENCY TEACUP AND SAUCER}),"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",0.036610,0.022893,0.020244,0.552972,24.154804,1.0,0.019406,2.185783,0.995028,0.515663,0.542498,0.718635
78,"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.022893,0.036610,0.020244,0.884298,24.154804,1.0,0.019406,8.326446,0.981060,0.515663,0.879901,0.718635
82,frozenset({PINK REGENCY TEACUP AND SAUCER}),"frozenset({ROSES REGENCY TEACUP AND SAUCER , G...",0.029799,0.028474,0.020244,0.679365,23.859031,1.0,0.019396,3.030006,0.987514,0.532338,0.669968,0.695164
79,"frozenset({ROSES REGENCY TEACUP AND SAUCER , G...",frozenset({PINK REGENCY TEACUP AND SAUCER}),0.028474,0.029799,0.020244,0.710963,23.859031,1.0,0.019396,3.356674,0.986167,0.532338,0.702086,0.695164
8,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.029799,0.036610,0.024596,0.825397,22.545917,1.0,0.023505,5.517600,0.984998,0.588235,0.818762,0.748616


In [208]:
!pip install lifetimes


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [209]:
from lifetimes import BetaGeoFitter
from lifetimes import GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

In [210]:
clv_data = summary_data_from_transaction_data(
    df,
    customer_id_col='CustomerID',
    datetime_col='InvoiceDate',
    monetary_value_col='Amount',
    observation_period_end=df['InvoiceDate'].max()
)

clv_data.head()


,frequency,recency,T,monetary_value
CustomerID,,,,
12346,0.0,0.0,316.0,0.000000
12347,1.0,278.0,308.0,1294.320068
12348,2.0,283.0,349.0,268.720001
12349,0.0,0.0,9.0,0.000000
12352,4.0,224.0,287.0,228.290009


In [211]:
bgf = BetaGeoFitter()
bgf.fit(clv_data['frequency'],
        clv_data['recency'],
        clv_data['T'])


C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


<lifetimes.BetaGeoFitter: fitted with 3480 subjects, a: 0.00, alpha: 99.21, b: 2112.95, r: 0.87>

In [212]:
clv_data['predicted_purchases'] = bgf.predict(
    30,
    clv_data['frequency'],
    clv_data['recency'],
    clv_data['T']
)


In [213]:
clv_data = clv_data[
    (clv_data['frequency'] > 0) &
    (clv_data['monetary_value'] > 0)
]

In [214]:
ggf = GammaGammaFitter()
ggf.fit(clv_data['frequency'],
        clv_data['monetary_value'])


<lifetimes.GammaGammaFitter: fitted with 1868 subjects, p: 2.23, q: 3.47, v: 484.34>

In [215]:
clv_data['predicted_clv'] = ggf.customer_lifetime_value(
    bgf,
    clv_data['frequency'],
    clv_data['recency'],
    clv_data['T'],
    clv_data['monetary_value'],
    time=6,  # 6 months prediction
    freq='D',
    discount_rate=0.01
)


In [216]:
clv_data.sort_values("predicted_clv", ascending=False).head()

,frequency,recency,T,monetary_value,predicted_purchases,predicted_clv
CustomerID,,,,,,
14646,30.0,338.0,345.0,6417.255859,2.084906,74961.590651
17450,19.0,250.0,251.0,7757.019531,1.702224,72538.582620
18102,11.0,248.0,250.0,10744.630859,1.019831,57921.069473
14096,10.0,90.0,92.0,4534.428223,1.705642,40775.883472
14911,85.0,349.0,349.0,987.556458,5.747619,32659.043908


In [217]:
df.to_csv("final_retail_full_data.csv", index=False)